# 05 · Feature Engineering

**Purpose:** Aggregate per `(ticker, date)` and create the predictive feature table.

**Base features (daily):**
- `news_count` — number of unique articles mentioning the ticker
- `avg_sentiment`, `std_sentiment` — mean and std of sentiment score
- `positive_ratio`, `negative_ratio` — share of positive / negative articles
- `event_ma`, `event_earnings`, `event_management`, `event_legal` — event mention counts
- `negated_count`, `speculative_count` — modifier counts

**Rolling features:** 3-day and 7-day rolling means for all base features.

**Input:** `articles_enriched.parquet`, `events.parquet`  
**Output:** `features.parquet`

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

NB_DIR = Path().resolve()
sys.path.insert(0, str(NB_DIR))
from utils import load, save

In [ ]:
enriched = load("articles_enriched")
events   = load("events")

print(f"Enriched rows: {len(enriched):,}")
print(f"Event rows:    {len(events):,}")

## 1 · Merge sentiment + events

In [ ]:
SENTIMENT_NUM = {"positive": 1.0, "neutral": 0.0, "negative": -1.0}

enriched = enriched.copy()
enriched["sentiment_num"] = enriched["sentiment"].map(SENTIMENT_NUM).fillna(0.0)
enriched["is_positive"]   = (enriched["sentiment"] == "positive").astype(int)
enriched["is_negative"]   = (enriched["sentiment"] == "negative").astype(int)

EVENT_BOOL_COLS = ["event_ma", "event_earnings", "event_management", "event_legal",
                   "is_negated", "is_speculative"]

merged = enriched.merge(
    events[["article_url", "ticker"] + EVENT_BOOL_COLS].drop_duplicates(["article_url", "ticker"]),
    on=["article_url", "ticker"],
    how="left",
)
for col in EVENT_BOOL_COLS:
    merged[col] = merged[col].fillna(False).astype(int)

merged["date"] = pd.to_datetime(merged["date"])
print(f"Merged rows: {len(merged):,}")

## 2 · Daily aggregation

In [ ]:
def aggregate_daily(df: pd.DataFrame) -> pd.DataFrame:
    agg = df.groupby(["ticker", "date"]).agg(
        news_count        =("article_url",   "nunique"),
        avg_sentiment     =("sentiment_num", "mean"),
        std_sentiment     =("sentiment_num", "std"),
        positive_count    =("is_positive",   "sum"),
        negative_count    =("is_negative",   "sum"),
        event_ma          =("event_ma",       "sum"),
        event_earnings    =("event_earnings", "sum"),
        event_management  =("event_management", "sum"),
        event_legal       =("event_legal",    "sum"),
        negated_count     =("is_negated",     "sum"),
        speculative_count =("is_speculative", "sum"),
    ).reset_index()

    agg["std_sentiment"]  = agg["std_sentiment"].fillna(0.0)
    agg["positive_ratio"] = agg["positive_count"] / agg["news_count"]
    agg["negative_ratio"] = agg["negative_count"] / agg["news_count"]
    return agg


daily = aggregate_daily(merged)
print(f"Daily feature rows: {len(daily):,}")
print(f"Tickers covered:    {daily['ticker'].nunique()}")
print(f"Date range:         {daily['date'].min().date()} → {daily['date'].max().date()}")
daily.head()

## 3 · Rolling features (3d, 7d)

In [ ]:
ROLLING_BASE_COLS = [
    "news_count", "avg_sentiment", "std_sentiment",
    "positive_ratio", "negative_ratio",
    "event_ma", "event_earnings", "event_management", "event_legal",
]

WINDOWS = [3, 7]


def add_rolling_features(df: pd.DataFrame, base_cols: list[str], windows: list[int]) -> pd.DataFrame:
    df = df.sort_values(["ticker", "date"]).copy()
    for w in windows:
        for col in base_cols:
            df[f"{col}_r{w}d"] = (
                df.groupby("ticker")[col]
                  .transform(lambda x: x.rolling(w, min_periods=1).mean())
            )
    return df


features = add_rolling_features(daily, ROLLING_BASE_COLS, WINDOWS)

feature_cols = [c for c in features.columns if c not in ["ticker", "date"]]
print(f"Total feature columns: {len(feature_cols)}")
print(", ".join(feature_cols))

## 4 · Quick visual check

In [ ]:
# Pick the ticker with the most news coverage for the plot
top_ticker = features.groupby("ticker")["news_count"].sum().idxmax()
ticker_df  = features[features["ticker"] == top_ticker].set_index("date")

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ticker_df["news_count"].plot(ax=axes[0], color="steelblue")
axes[0].set_title(f"{top_ticker} — daily news count")

ticker_df[["avg_sentiment", "avg_sentiment_r7d"]].plot(ax=axes[1])
axes[1].axhline(0, color="black", linewidth=0.5, linestyle="--")
axes[1].set_title(f"{top_ticker} — avg sentiment (raw vs 7d rolling)")

plt.tight_layout()
plt.show()

In [ ]:
save(features, "features")